In [1]:
import os
from dotenv import load_dotenv
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import asyncio

from benchmarklib import BenchmarkDatabase
from sat import ThreeSat, ThreeSatTrial, ThreeSatOracleTrial
# from benchmarklib.compilers import CompileType, XAGCompiler

from qiskit_ibm_runtime import QiskitRuntimeService

import logging
from typing import Iterable, List, Tuple, Dict, Any, Union, Optional
import qiskit
from qiskit.providers import Backend
from qiskit import QuantumCircuit, transpile
import random

from sqlalchemy import select, func

from tweedledum.bool_function_compiler import circuit_input, QuantumCircuitFunction
from tweedledum import BitVec

from benchmarklib import CompileType, BenchmarkDatabase
from benchmarklib import BatchQueue
from benchmarklib.compilers import SynthesisCompiler

In [2]:
benchmark_db = BenchmarkDatabase("three_sat.db", ThreeSat, ThreeSatTrial)

In [3]:
from benchmarklib.algorithms.grover import build_grover_circuit, calculate_grover_iterations,  verify_oracle

In [8]:
relevant_problem_ids = benchmark_db.query(select(ThreeSat.id).select_from(ThreeSatTrial).join(ThreeSatTrial.problem))
for problem in benchmark_db.query(select(ThreeSat).where(ThreeSat.id.in_(relevant_problem_ids))):
    optimal_grover_iters = calculate_grover_iterations(len(problem.solutions), 2**problem.num_vars)
    if optimal_grover_iters > 1:
        print(f"Problem {problem.id} has {len(problem.solutions)} solutions and {problem.num_vars} variables, optimal Grover iterations: {optimal_grover_iters}")
        
    


Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 2 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 31 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 38 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 41 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 52 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=0.125, angle=0.361, result=2.173, floor=2 : 0.945, round=3 : 0.330
Problem 53 has 1 solutions and 3 variables, optimal Grover iterations: 2
Debug: M=1, N=8, ratio=

In [9]:
problems = benchmark_db.query(select(ThreeSat).where(ThreeSat.id.in_([2, 123, 210])))

In [10]:
for problem in problems:
    print(problem.expr)

[[1, 2, 3], [2, -1, 3], [-2, 3, 1], [-1, 3, 2], [-2, -1, -3], [-2, -1, -3], [-2, -1, 3], [-3, -2, 1], [-2, 3, 1], [2, -3, -1], [3, -2, -1], [3, 2, 1], [3, 2, 1], [-1, -3, -2], [2, -3, -1]]
[[2, 4, -1], [-4, 2, -1], [-1, 2, 4], [-4, -2, -3], [-3, -2, 4], [4, -1, 3], [-3, 2, 4], [3, 2, -1], [-2, -1, -4], [-3, 4, -1], [2, -4, -1], [-3, 2, -4], [-1, -2, -4], [4, 2, -1], [1, -2, -3], [1, 3, 2], [-3, -4, 1], [3, -1, 2], [-1, 4, 3], [-1, -4, 2], [3, 2, 4]]
[[-1, -3, 5], [2, 5, 3], [-3, -2, -5], [-1, -3, -5], [2, 3, 4], [1, -3, -4], [-3, 2, -4], [-1, 4, -3], [-5, -1, -4], [1, 3, 4], [3, -5, -2], [-4, 2, -3], [-2, -3, -1], [-2, 5, -1], [-5, 3, -2], [1, -5, 4], [1, -2, 5], [5, 1, -4], [-2, -5, 1], [-5, 1, 4], [-2, 4, 5], [-3, -2, 5], [-2, -5, -4], [-3, -4, -1], [1, 2, -4], [3, -1, -5], [-4, -1, -2], [4, -2, 1], [4, 3, -2]]


In [ ]:
calculate_grover_iterations(len(problem.solutions), 2**num_vars)

In [ ]:
verify_oracle()

In [12]:
trials = benchmark_db.query(
    select(ThreeSatTrial).join(ThreeSatTrial.problem).where(ThreeSatTrial.grover_iters > 1).limit(10)
)

In [18]:
benchmark_db.query(
    select(func.count(ThreeSatTrial.id)).where(ThreeSatTrial.grover_iters >= 0)
)

[0]

In [13]:
for trial in trials:
    print(trial.problem.g, trial.grover_iters, trial.compiler_type, trial.compile_time, trial.circuit_depth, trial.circuit_width)

In [3]:
problems = benchmark_db.query(select(ThreeSat).where(ThreeSat.num_vars == 4).limit(10))

In [ ]:
for problem in problems:
    print(len(problem.solutions), problem.g)

1
3
1
3
2
1
2
3
2
2
